# 🍎🍊🍌 Fruit Quality Grading
## Step 1: Feature Extraction + K-Means Labeling

This notebook extracts visual features from fresh fruit images and uses **K-Means clustering** to automatically assign quality grades (**A, B, C**) to each image — replacing the need for manual labeling.

### Pipeline
```
Fresh fruit images
       ↓
Extract features (Color → Texture → Shape)
       ↓
Normalize features (StandardScaler)
       ↓
K-Means clustering per fruit (k=3)
       ↓
Auto-assign Grade A / B / C
       ↓
Save labeled_dataset.csv  ← input for CNN training
```

### Fruits
- 🍎 Apple
- 🍊 Orange  
- 🍌 Banana

### Expected Dataset Structure (Kaggle Fresh & Rotten)
```
dataset/
├── train/
│   ├── freshapples/
│   ├── freshoranges/
│   └── freshbananas/
└── test/
    ├── freshapples/
    ├── freshoranges/
    └── freshbananas/
```

## 📦 1. Install Dependencies

In [ ]:
!pip install opencv-python scikit-learn scikit-image pandas tqdm matplotlib -q

## 📚 2. Imports

In [ ]:
import os
import shutil
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from skimage.feature import graycomatrix, graycoprops
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
print("✅ All libraries imported successfully")

## ⚙️ 3. Configuration

> **Edit `DATASET_ROOT`** to point to where you downloaded the Kaggle dataset.

In [ ]:
# ─────────────────────────────────────────────
#  CONFIGURATION — edit these as needed
# ─────────────────────────────────────────────

DATASET_ROOT  = "dataset"    # ← path to your Kaggle dataset folder
OUTPUT_DIR    = "output"     # ← where results will be saved
IMAGE_SIZE    = (224, 224)   # resize all images before feature extraction
N_CLUSTERS    = 3            # number of quality grades (A, B, C)
N_SAMPLES     = 10           # sample images saved per cluster for inspection
RANDOM_STATE  = 42

# Fruit folder names in the dataset → clean label
FRUIT_FOLDERS = {
    "freshapples":  "apple",
    "freshoranges": "orange",
    "freshbananas": "banana",
}

# ── Manual grade override (optional) ──────────────────────────────
# After running the notebook, open output/cluster_samples/ and inspect
# the sample images. If the auto-assignment looks wrong, set the correct
# mapping here and re-run from Cell 7 onwards.
#
# Example:
# GRADE_OVERRIDE = {
#     "apple":  {0: "B", 1: "A", 2: "C"},
#     "orange": {0: "A", 1: "C", 2: "B"},
#     "banana": {0: "C", 1: "B", 2: "A"},
# }
GRADE_OVERRIDE = {}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"✅ Config set — output will be saved to: {os.path.abspath(OUTPUT_DIR)}")

## 🔬 4. Feature Extraction Functions

We extract **18 features** per image across 3 groups:

| Group | Features | Quality Signal |
|---|---|---|
| **Color** | Hue, Saturation, Brightness (mean & std), Color uniformity, RGB means | Vibrant, uniform color = higher grade |
| **Texture** | GLCM Contrast, Homogeneity, Energy, Correlation, Dissimilarity | Smooth surface = higher grade |
| **Shape** | Blemish ratio, Edge density, Roundness | Fewer marks, rounder = higher grade |

In [ ]:
def load_image(path: str):
    """Load and resize an image. Returns None if loading fails."""
    img = cv2.imread(path)
    if img is None:
        return None
    return cv2.resize(img, IMAGE_SIZE)


def extract_color_features(img_bgr: np.ndarray) -> dict:
    """Extract HSV and RGB color features."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    h, s, v = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]
    r, g, b = rgb[:,:,0], rgb[:,:,1], rgb[:,:,2]
    return {
        "hue_mean":         np.mean(h),
        "hue_std":          np.std(h),
        "sat_mean":         np.mean(s),
        "sat_std":          np.std(s),
        "val_mean":         np.mean(v),  # brightness
        "val_std":          np.std(v),
        "color_uniformity": 1.0 / (np.std(s) + 1e-5),
        "r_mean":           np.mean(r),
        "g_mean":           np.mean(g),
        "b_mean":           np.mean(b),
    }


def extract_texture_features(img_bgr: np.ndarray) -> dict:
    """Extract GLCM texture features."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    gray_small = cv2.resize(gray, (64, 64))
    glcm = graycomatrix(
        gray_small,
        distances=[1, 3],
        angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256, symmetric=True, normed=True,
    )
    return {
        "texture_contrast":      graycoprops(glcm, "contrast").mean(),
        "texture_dissimilarity": graycoprops(glcm, "dissimilarity").mean(),
        "texture_homogeneity":   graycoprops(glcm, "homogeneity").mean(),
        "texture_energy":        graycoprops(glcm, "energy").mean(),
        "texture_correlation":   graycoprops(glcm, "correlation").mean(),
    }


def extract_shape_features(img_bgr: np.ndarray) -> dict:
    """Extract blemish, edge density, and roundness features."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # Blemish ratio: dark pixels relative to mean brightness
    blemish_ratio = np.sum(gray < np.mean(gray) * 0.5) / gray.size

    # Edge density via Canny
    edges = cv2.Canny(gray, 50, 150)
    edge_density = np.sum(edges > 0) / edges.size

    # Roundness via largest contour
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    roundness = 0.0
    if contours:
        largest = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(largest)
        perimeter = cv2.arcLength(largest, True)
        if perimeter > 0:
            roundness = (4 * np.pi * area) / (perimeter ** 2)

    return {
        "blemish_ratio": blemish_ratio,
        "edge_density":  edge_density,
        "roundness":     roundness,
    }


def extract_all_features(img_bgr: np.ndarray) -> dict:
    """Combine all feature groups into one feature vector."""
    feats = {}
    feats.update(extract_color_features(img_bgr))
    feats.update(extract_texture_features(img_bgr))
    feats.update(extract_shape_features(img_bgr))
    return feats


print("✅ Feature extraction functions defined")

## 🛠️ 5. Helper Functions (Data Collection, Grading, Visualization)

In [ ]:
def collect_image_paths(dataset_root: str) -> dict:
    """Scan dataset folder and collect image paths per fruit type."""
    root = Path(dataset_root)
    fruit_paths = {fruit: [] for fruit in FRUIT_FOLDERS.values()}

    for split in ["train", "test"]:
        for folder_name, fruit_label in FRUIT_FOLDERS.items():
            folder = root / split / folder_name
            if not folder.exists():
                folder = root / folder_name  # try without split subfolder
            if folder.exists():
                paths = (list(folder.glob("*.jpg")) +
                         list(folder.glob("*.jpeg")) +
                         list(folder.glob("*.png")))
                fruit_paths[fruit_label].extend([str(p) for p in paths])

    return fruit_paths


def auto_assign_grades(cluster_centers: np.ndarray, feature_names: list) -> dict:
    """
    Rank clusters by a quality score to assign A / B / C.

    Quality score weights:
    - Brightness (val_mean)       +0.25  → brighter = fresher
    - Saturation (sat_mean)       +0.20  → more color = riper
    - Color uniformity            +0.20  → uniform = consistent quality
    - Texture homogeneity         +0.15  → smooth surface = higher grade
    - Blemish ratio               -0.10  → fewer blemishes = better
    - Edge density                -0.10  → smoother surface = better
    """
    idx = {name: i for i, name in enumerate(feature_names)}
    scores = []
    for center in cluster_centers:
        score = (
            + 0.25 * center[idx.get("val_mean", 0)]           / 255.0
            + 0.20 * center[idx.get("sat_mean", 0)]           / 255.0
            + 0.20 * center[idx.get("color_uniformity", 0)]   / 10.0
            + 0.15 * center[idx.get("texture_homogeneity", 0)]
            - 0.10 * center[idx.get("blemish_ratio", 0)]
            - 0.10 * center[idx.get("edge_density", 0)]
        )
        scores.append(score)

    ranked = np.argsort(scores)[::-1]  # highest score = Grade A
    return {int(cluster_id): ["A", "B", "C"][rank] for rank, cluster_id in enumerate(ranked)}


def save_cluster_samples(image_paths, cluster_labels, fruit, n_samples=10):
    """Copy sample images from each cluster for manual inspection."""
    for cluster_id in range(N_CLUSTERS):
        out_dir = Path(OUTPUT_DIR) / "cluster_samples" / f"{fruit}_cluster_{cluster_id}"
        out_dir.mkdir(parents=True, exist_ok=True)
        indices = np.where(cluster_labels == cluster_id)[0]
        sampled = np.random.choice(indices, size=min(n_samples, len(indices)), replace=False)
        for i, idx in enumerate(sampled):
            src = image_paths[idx]
            dst = out_dir / f"sample_{i:02d}{Path(src).suffix}"
            shutil.copy2(src, dst)


def plot_pca_clusters(X_scaled, cluster_labels, grade_map, fruit):
    """Reduce to 2D with PCA and plot the quality clusters."""
    pca    = PCA(n_components=2, random_state=RANDOM_STATE)
    coords = pca.fit_transform(X_scaled)

    colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}
    point_colors = [colors[grade_map[lbl]] for lbl in cluster_labels]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(coords[:, 0], coords[:, 1], c=point_colors, alpha=0.5, s=15, edgecolors="none")

    patches = [mpatches.Patch(color=colors[g], label=f"Grade {g}") for g in ["A", "B", "C"]]
    ax.legend(handles=patches, fontsize=12)

    var = pca.explained_variance_ratio_
    ax.set_xlabel(f"PC1 ({var[0]*100:.1f}% variance)", fontsize=11)
    ax.set_ylabel(f"PC2 ({var[1]*100:.1f}% variance)", fontsize=11)
    ax.set_title(f"{fruit.capitalize()} — K-Means Quality Clusters (PCA 2D)", fontsize=13, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.4)

    plots_dir = Path(OUTPUT_DIR) / "plots"
    plots_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(plots_dir / f"{fruit}_pca.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  📊 Plot saved → output/plots/{fruit}_pca.png")


print("✅ Helper functions defined")

## 📁 6. Collect Image Paths

In [ ]:
fruit_paths = collect_image_paths(DATASET_ROOT)

print("Images found per fruit:")
for fruit, paths in fruit_paths.items():
    status = "✅" if paths else "❌ NOT FOUND"
    print(f"  {fruit:8s} → {len(paths):,} images  {status}")

if all(len(v) == 0 for v in fruit_paths.values()):
    print(f"\n⚠️  No images found. Check that DATASET_ROOT = '{DATASET_ROOT}' is correct.")
    print(f"   Full path: {os.path.abspath(DATASET_ROOT)}")

## 🔍 7. Extract Features from All Images

This cell loops through every fruit image and extracts the 18 visual features.  
It may take a few minutes depending on dataset size.

In [ ]:
fruit_feature_data = {}   # stores DataFrame of features per fruit
fruit_valid_paths  = {}   # stores valid image paths per fruit

for fruit, paths in fruit_paths.items():
    if not paths:
        print(f"⚠️  Skipping {fruit} — no images found.")
        continue

    print(f"\n🔍 Extracting features: {fruit.upper()} ({len(paths):,} images)...")

    feature_rows = []
    valid_paths  = []

    for path in tqdm(paths, desc=f"  {fruit}"):
        img = load_image(path)
        if img is None:
            continue
        feats = extract_all_features(img)
        feats["image_path"] = path
        feats["fruit"]      = fruit
        feature_rows.append(feats)
        valid_paths.append(path)

    df = pd.DataFrame(feature_rows)
    fruit_feature_data[fruit] = df
    fruit_valid_paths[fruit]  = valid_paths
    print(f"  ✅ {len(valid_paths):,} images processed | {len(df.columns)-2} features extracted")

# Save raw features
all_features_df = pd.concat(fruit_feature_data.values(), ignore_index=True)
all_features_df.to_csv(Path(OUTPUT_DIR) / "kmeans_features.csv", index=False)
print(f"\n💾 Raw features saved → output/kmeans_features.csv")

## 👀 8. Preview Extracted Features

In [ ]:
# Show a sample of extracted features for one fruit
sample_fruit = list(fruit_feature_data.keys())[0]
feature_cols = [c for c in fruit_feature_data[sample_fruit].columns if c not in ("image_path", "fruit")]

print(f"Feature preview — {sample_fruit.upper()} (first 5 rows):\n")
fruit_feature_data[sample_fruit][feature_cols].head()

## 📊 9. Feature Distribution (Key Quality Features)

In [ ]:
key_features = ["val_mean", "sat_mean", "color_uniformity", "blemish_ratio",
                "texture_homogeneity", "edge_density", "roundness"]

fig, axes = plt.subplots(len(key_features), 1, figsize=(10, 3 * len(key_features)))
fruit_colors = {"apple": "#e74c3c", "orange": "#f39c12", "banana": "#f1c40f"}

for i, feat in enumerate(key_features):
    ax = axes[i]
    for fruit, df in fruit_feature_data.items():
        if feat in df.columns:
            ax.hist(df[feat], bins=40, alpha=0.5, label=fruit, color=fruit_colors.get(fruit))
    ax.set_title(feat.replace("_", " ").title(), fontsize=11, fontweight="bold")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.3)

plt.suptitle("Feature Distributions per Fruit", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "plots" / "feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 🤖 10. K-Means Clustering (per Fruit)

Each fruit is clustered **independently** so K-Means finds quality variation *within* that fruit type, not differences between fruit types.

In [ ]:
all_records    = []   # will become the final labeled CSV
fruit_clusters = {}   # stores cluster labels per fruit

for fruit, df in fruit_feature_data.items():
    print(f"\n{'─'*50}")
    print(f"🍎 Clustering: {fruit.upper()}")
    print(f"{'─'*50}")

    feature_cols  = [c for c in df.columns if c not in ("image_path", "fruit")]
    X             = df[feature_cols].values
    valid_paths   = fruit_valid_paths[fruit]

    # Normalize
    scaler   = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # K-Means
    kmeans        = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20, max_iter=500)
    cluster_labels = kmeans.fit_predict(X_scaled)

    # Grade assignment
    if fruit in GRADE_OVERRIDE:
        grade_map = GRADE_OVERRIDE[fruit]
        print(f"  ℹ️  Using manual override: {grade_map}")
    else:
        grade_map = auto_assign_grades(kmeans.cluster_centers_, feature_cols)
        print(f"  🤖 Auto-assigned grades:   {grade_map}")

    # Print distribution
    unique, counts = np.unique(cluster_labels, return_counts=True)
    print("  Cluster distribution:")
    for cid, cnt in zip(unique, counts):
        grade = grade_map[int(cid)]
        pct   = cnt / len(cluster_labels) * 100
        bar   = "█" * int(pct / 2)
        print(f"    Cluster {cid} → Grade {grade}: {cnt:>5,} images ({pct:5.1f}%)  {bar}")

    # Save cluster samples for manual inspection
    save_cluster_samples(valid_paths, cluster_labels, fruit, N_SAMPLES)

    # Store for PCA plot
    fruit_clusters[fruit] = (X_scaled, cluster_labels, grade_map, feature_cols)

    # Collect labeled records
    for i, path in enumerate(valid_paths):
        cid   = int(cluster_labels[i])
        grade = grade_map[cid]
        all_records.append({
            "image_path": path,
            "fruit":      fruit,
            "cluster_id": cid,
            "grade":      grade,
            "label":      f"{fruit}_{grade}",
        })

print("\n✅ Clustering complete for all fruits")

## 📈 11. PCA Cluster Visualizations

These 2D plots show how well K-Means separated the quality clusters. Well-separated clusters = reliable grade labels.

In [ ]:
for fruit, (X_scaled, cluster_labels, grade_map, _) in fruit_clusters.items():
    print(f"\n📊 PCA plot: {fruit.upper()}")
    plot_pca_clusters(X_scaled, cluster_labels, grade_map, fruit)

## 🖼️ 12. Visual Cluster Inspection

**This is the most important manual step.**

Look at the sample images below for each cluster and verify:
- **Grade A** → vibrant color, smooth surface, no marks
- **Grade B** → slightly dull or minor imperfections  
- **Grade C** → dull color, visible marks, irregular shape

If the grading looks wrong → update `GRADE_OVERRIDE` in Cell 3 and re-run from Cell 10.

In [ ]:
def show_cluster_samples(fruit, grade_map, n_show=5):
    """Display sample images from each cluster inline."""
    fig, axes = plt.subplots(N_CLUSTERS, n_show, figsize=(3 * n_show, 3.5 * N_CLUSTERS))
    grade_colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}

    for row, cluster_id in enumerate(range(N_CLUSTERS)):
        grade      = grade_map[cluster_id]
        sample_dir = Path(OUTPUT_DIR) / "cluster_samples" / f"{fruit}_cluster_{cluster_id}"
        images     = list(sample_dir.glob("*.jpg")) + list(sample_dir.glob("*.png"))[:n_show]

        for col in range(n_show):
            ax = axes[row, col]
            ax.axis("off")
            if col < len(images):
                img = cv2.imread(str(images[col]))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                if col == 0:
                    ax.set_title(f"Grade {grade}", fontsize=13, fontweight="bold",
                                 color=grade_colors[grade], pad=8)

    plt.suptitle(f"{fruit.capitalize()} — Cluster Samples (verify grades below)",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "plots" / f"{fruit}_cluster_samples.png",
                dpi=120, bbox_inches="tight")
    plt.show()


for fruit, (_, _, grade_map, _) in fruit_clusters.items():
    print(f"\n{'='*50}")
    print(f"  {fruit.upper()} — Visual Grade Check")
    print(f"{'='*50}")
    show_cluster_samples(fruit, grade_map)

## 💾 13. Save Labeled Dataset

In [ ]:
df_labeled = pd.DataFrame(all_records)

labeled_csv = Path(OUTPUT_DIR) / "labeled_dataset.csv"
df_labeled.to_csv(labeled_csv, index=False)

print(f"✅ Labeled dataset saved → {labeled_csv}")
print(f"   Total images: {len(df_labeled):,}\n")
print("Grade distribution:")
dist = df_labeled["label"].value_counts().sort_index()
for label, count in dist.items():
    bar = "█" * (count // 50)
    print(f"  {label:12s}  {count:>5,}  {bar}")

## 🔍 14b. Boundary Detection — Flag Borderline Images

K-Means assigns every image to a cluster, but images **near the boundary between two clusters** are the most likely to be mislabeled. These are the ones where the model is least certain — a fruit that is visually halfway between Grade A and Grade B.

### What this cell does
1. Computes each image's **distance to its own cluster center** vs its **nearest other cluster center**
2. Calculates a **boundary score** — how close it is to being assigned a different grade
3. Flags the top borderline images per fruit for your manual review
4. Saves them to `output/boundary_review/` so you can relabel if needed

### Boundary Score
```
boundary_score = distance_to_own_center / distance_to_nearest_other_center
```
- Score close to **1.0** = right on the boundary (most uncertain)
- Score close to **0.0** = deep inside its cluster (most certain)

**You only need to review ~50–100 images**, not the full dataset.

In [ ]:
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist

BOUNDARY_THRESHOLD = 0.85   # images with score >= this are flagged as borderline
N_BOUNDARY_SAMPLES = 15     # max samples to copy per cluster per fruit for review

boundary_dir = Path(OUTPUT_DIR) / "boundary_review"
boundary_dir.mkdir(parents=True, exist_ok=True)

all_boundary_records = []

for fruit, df_feat in fruit_feature_data.items():
    print(f"\n{'─'*50}")
    print(f"  Boundary detection: {fruit.upper()}")
    print(f"{'─'*50}")

    feature_cols = [c for c in df_feat.columns if c not in ("image_path", "fruit")]
    X            = df_feat[feature_cols].values
    valid_paths  = fruit_valid_paths[fruit]

    # Re-run K-Means with same seed to get cluster centers
    scaler    = StandardScaler()
    X_scaled  = scaler.fit_transform(X)
    kmeans    = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20, max_iter=500)
    labels    = kmeans.fit_predict(X_scaled)
    centers   = kmeans.cluster_centers_

    # Get grade map from previously computed fruit_clusters
    _, _, grade_map, _ = fruit_clusters[fruit]

    # Compute distances from each point to ALL cluster centers
    # distances shape: (n_images, n_clusters)
    distances = cdist(X_scaled, centers, metric="euclidean")

    boundary_scores = []
    for i in range(len(X_scaled)):
        own_cluster   = labels[i]
        dist_own      = distances[i, own_cluster]

        # Distance to nearest OTHER cluster
        other_dists   = [distances[i, c] for c in range(N_CLUSTERS) if c != own_cluster]
        dist_other    = min(other_dists)

        # Score: ratio of own distance to nearest other distance
        # High score = close to boundary
        score = dist_own / (dist_other + 1e-8)
        boundary_scores.append(score)

    boundary_scores = np.array(boundary_scores)

    # Flag borderline images
    is_borderline = boundary_scores >= BOUNDARY_THRESHOLD
    n_borderline  = is_borderline.sum()
    pct           = n_borderline / len(boundary_scores) * 100

    print(f"  Total images     : {len(boundary_scores):,}")
    print(f"  Borderline (>={BOUNDARY_THRESHOLD}) : {n_borderline:,}  ({pct:.1f}%)")

    # Distribution per cluster
    print("  Borderline per cluster:")
    for cid in range(N_CLUSTERS):
        grade    = grade_map[cid]
        in_clust = (labels == cid)
        n_border = (is_borderline & in_clust).sum()
        n_total  = in_clust.sum()
        print(f"    Cluster {cid} (Grade {grade}): {n_border:>4} / {n_total:<5} borderline")

    # Save borderline samples for manual review
    fruit_boundary_dir = boundary_dir / fruit
    fruit_boundary_dir.mkdir(exist_ok=True)

    # Sort by boundary score (highest = most uncertain)
    borderline_indices = np.where(is_borderline)[0]
    borderline_indices = borderline_indices[np.argsort(boundary_scores[borderline_indices])[::-1]]

    copied = 0
    for idx in borderline_indices[:N_BOUNDARY_SAMPLES * N_CLUSTERS]:
        src   = valid_paths[idx]
        cid   = int(labels[idx])
        grade = grade_map[cid]
        score = boundary_scores[idx]
        dst   = fruit_boundary_dir / f"score{score:.3f}_cluster{cid}_grade{grade}_{Path(src).name}"
        shutil.copy2(src, dst)
        copied += 1

        # Also track in records
        all_boundary_records.append({
            "image_path":     src,
            "fruit":          fruit,
            "cluster_id":     cid,
            "assigned_grade": grade,
            "boundary_score": round(score, 4),
            "label":          f"{fruit}_{grade}",
        })

    print(f"  Saved {copied} borderline samples -> output/boundary_review/{fruit}/")

# Save boundary records CSV
df_boundary = pd.DataFrame(all_boundary_records).sort_values(
    ["fruit", "boundary_score"], ascending=[True, False]
).reset_index(drop=True)

df_boundary.to_csv(Path(OUTPUT_DIR) / "boundary_images.csv", index=False)
print(f"\nBoundary records saved -> output/boundary_images.csv")
print(f"Total flagged: {len(df_boundary):,} images")


## 📊 14c. Boundary Score Distribution

Visualize how many images are near cluster boundaries per fruit.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fruit_colors = {"apple": "#e74c3c", "orange": "#f39c12", "banana": "#f1c40f"}

for ax, (fruit, df_feat) in zip(axes, fruit_feature_data.items()):
    feature_cols = [c for c in df_feat.columns if c not in ("image_path", "fruit")]
    X            = df_feat[feature_cols].values
    scaler       = StandardScaler()
    X_scaled     = scaler.fit_transform(X)
    kmeans       = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=20, max_iter=500)
    labels       = kmeans.fit_predict(X_scaled)
    centers      = kmeans.cluster_centers_
    distances    = cdist(X_scaled, centers, metric="euclidean")

    scores = []
    for i in range(len(X_scaled)):
        own   = distances[i, labels[i]]
        other = min(distances[i, c] for c in range(N_CLUSTERS) if c != labels[i])
        scores.append(own / (other + 1e-8))
    scores = np.array(scores)

    ax.hist(scores, bins=40, color=fruit_colors[fruit], edgecolor="white", alpha=0.85)
    ax.axvline(x=BOUNDARY_THRESHOLD, color="#e74c3c", linestyle="--", linewidth=2,
               label=f"Threshold ({BOUNDARY_THRESHOLD})")
    n_flagged = (scores >= BOUNDARY_THRESHOLD).sum()
    ax.set_title(f"{fruit.capitalize()}\n{n_flagged} borderline ({n_flagged/len(scores)*100:.1f}%)",
                 fontsize=12, fontweight="bold")
    ax.set_xlabel("Boundary Score")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9)
    ax.grid(True, linestyle="--", alpha=0.3)

plt.suptitle("Boundary Score Distribution per Fruit\n(higher score = closer to cluster boundary = more uncertain)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "plots" / "boundary_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> output/plots/boundary_scores.png")


## 🖼️ 14d. View Borderline Images for Manual Review

These are the images the algorithm was **most uncertain about**. Review them and decide if the assigned grade looks correct.

### How to relabel
1. Look at the images in `output/boundary_review/<fruit>/`
2. The filename includes the assigned grade e.g. `score0.923_cluster1_gradeA_img001.jpg`
3. If you disagree with the grade, note the `image_path` and your preferred grade
4. Run the **correction cell below** (14e) to apply your fixes to `labeled_dataset.csv`

In [ ]:
def show_borderline_samples(fruit, n_show=10):
    """Display the most borderline images for a fruit."""
    review_dir = Path(OUTPUT_DIR) / "boundary_review" / fruit
    if not review_dir.exists():
        print(f"No boundary samples found for {fruit}")
        return

    images = sorted(review_dir.glob("*.jpg")) + sorted(review_dir.glob("*.png"))
    images = images[:n_show]
    if not images:
        return

    cols = min(5, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows))
    axes = np.array(axes).flatten()

    grade_colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}

    for i, (ax, img_path) in enumerate(zip(axes, images)):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (112, 112))
        ax.imshow(img)
        ax.axis("off")
        # Parse score and grade from filename
        parts = img_path.stem.split("_")
        score = parts[0].replace("score", "")
        grade = parts[2].replace("grade", "")
        ax.set_title(f"Grade {grade}\nscore: {score}",
                     fontsize=9, color=grade_colors.get(grade, "black"),
                     fontweight="bold")

    for ax in axes[len(images):]:
        ax.axis("off")

    plt.suptitle(f"{fruit.capitalize()} — Most Borderline Images (review these)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(Path(OUTPUT_DIR) / "plots" / f"{fruit}_borderline_samples.png",
                dpi=120, bbox_inches="tight")
    plt.show()


for fruit in fruit_feature_data.keys():
    print(f"\n{'='*50}")
    print(f"  {fruit.upper()} — Borderline Samples")
    print(f"{'='*50}")
    show_borderline_samples(fruit, n_show=10)


## ✏️ 14e. Apply Manual Corrections

After reviewing the borderline images, enter your corrections here.
Only correct images you are **confident** about — leave uncertain ones as-is.

In [ ]:
# ─────────────────────────────────────────────────────────
#  MANUAL CORRECTIONS
#  Add entries as: "path/to/image.jpg": "A"  (or "B" or "C")
#  Leave empty if no corrections needed.
# ─────────────────────────────────────────────────────────

corrections = {
    # Example (remove these and add your own):
    # "dataset/train/freshapples/img_001.jpg": "B",
    # "dataset/train/freshoranges/img_045.jpg": "A",
}

# ── Apply corrections ──────────────────────────────────────
if corrections:
    df_corrected = df_labeled.copy()
    n_changed = 0

    for img_path, new_grade in corrections.items():
        mask = df_corrected["image_path"] == img_path
        if mask.sum() == 0:
            print(f"  WARNING: path not found in dataset: {img_path}")
            continue
        old_grade = df_corrected.loc[mask, "grade"].values[0]
        fruit     = df_corrected.loc[mask, "fruit"].values[0]
        df_corrected.loc[mask, "grade"] = new_grade
        df_corrected.loc[mask, "label"] = f"{fruit}_{new_grade}"
        print(f"  Corrected: {Path(img_path).name}  {old_grade} -> {new_grade}")
        n_changed += 1

    # Overwrite labeled_dataset.csv
    df_corrected.to_csv(Path(OUTPUT_DIR) / "labeled_dataset.csv", index=False)
    df_labeled = df_corrected  # update in memory too

    print(f"\n{n_changed} corrections applied.")
    print("labeled_dataset.csv updated.")
else:
    print("No corrections entered — labeled_dataset.csv unchanged.")
    print("If you want to correct labels, add entries to the 'corrections' dict above.")


## 📋 14f. Boundary Review Summary

A quick summary of label confidence across the full dataset.

In [ ]:
print("=" * 55)
print("  BOUNDARY REVIEW SUMMARY")
print("=" * 55)

total_images    = len(df_labeled)
total_borderline = len(df_boundary)
pct_borderline  = total_borderline / total_images * 100

print(f"  Total labeled images   : {total_images:,}")
print(f"  Borderline images      : {total_borderline:,}  ({pct_borderline:.1f}%)")
print(f"  High-confidence images : {total_images - total_borderline:,}  ({100-pct_borderline:.1f}%)")
print()
print("  Borderline breakdown per fruit:")
for fruit in fruit_feature_data.keys():
    n_fruit    = len(df_labeled[df_labeled["fruit"] == fruit])
    n_border   = len(df_boundary[df_boundary["fruit"] == fruit])
    print(f"    {fruit:8s}  {n_border:>4} / {n_fruit:<5}  ({n_border/n_fruit*100:.1f}%)")

print()
print("  Files saved:")
print("    output/boundary_review/<fruit>/  <- images to inspect")
print("    output/boundary_images.csv       <- full borderline records")
print("    output/plots/boundary_scores.png <- score distribution")
print()
print("  Next steps:")
print("    1. Review images in output/boundary_review/")
print("    2. Add corrections to Cell 14e if needed")
print("    3. Proceed to Step 2 CNN Training")


## 👀 14. Preview Labeled Dataset

In [ ]:
print(f"Shape: {df_labeled.shape}")
print(f"Columns: {list(df_labeled.columns)}\n")
df_labeled.head(10)

## 📊 15. Grade Distribution Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Chart 1: Per fruit per grade ──
fruits = df_labeled["fruit"].unique()
grades = ["A", "B", "C"]
x      = np.arange(len(fruits))
width  = 0.25
colors = {"A": "#2ecc71", "B": "#f39c12", "C": "#e74c3c"}

for i, grade in enumerate(grades):
    counts = [len(df_labeled[(df_labeled["fruit"] == f) & (df_labeled["grade"] == grade)]) for f in fruits]
    axes[0].bar(x + i * width, counts, width, label=f"Grade {grade}", color=colors[grade], edgecolor="white")

axes[0].set_xticks(x + width)
axes[0].set_xticklabels([f.capitalize() for f in fruits], fontsize=12)
axes[0].set_title("Images per Fruit per Grade", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Images")
axes[0].legend(fontsize=11)
axes[0].grid(True, axis="y", linestyle="--", alpha=0.4)

# ── Chart 2: Overall grade distribution ──
overall = df_labeled["grade"].value_counts().sort_index()
axes[1].bar(overall.index, overall.values,
            color=[colors[g] for g in overall.index], edgecolor="white", width=0.5)
for i, (grade, cnt) in enumerate(overall.items()):
    axes[1].text(i, cnt + 20, str(cnt), ha="center", fontsize=12, fontweight="bold")
axes[1].set_title("Overall Grade Distribution", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Images")
axes[1].set_xlabel("Grade")
axes[1].grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "plots" / "grade_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved → output/plots/grade_distribution.png")

## ✅ 16. Summary & Next Steps

In [ ]:
print("=" * 60)
print("  STEP 1 COMPLETE — Feature Extraction + K-Means Labeling")
print("=" * 60)
print("Output saved to output/labeled_dataset.csv")
print(f"Total images  : {len(df_labeled):,}")
print(f"Fruits        : {', '.join(df_labeled['fruit'].unique())}")
print(f"Grades        : A, B, C")
print(f"Output labels : {', '.join(sorted(df_labeled['label'].unique()))}")
print("")
print("Before moving to CNN training:")
print("  1. Open output/cluster_samples/ and verify the grade assignments")
print("  2. If grades look wrong, update GRADE_OVERRIDE in Cell 3 and re-run from Cell 10")
print("  3. Once satisfied, use labeled_dataset.csv as input for the CNN")
print("")
print("Next: Step 2 - CNN Training (EfficientNetB0)")